In [40]:
import os, json
import pandas as pd


# analyse the steering results
success schemas:
- counting 1s and -1s
    - counts of 1s of 2pos
    - counts of -1s of 2neg
- comparing the baseline
    - if baseline is 1, after 2neg, how many becomes -1 or 0?
    - if baseline is 0, 
        - after 2neg, how many becomes -1?
        - after 2pos, how many becomes 1?
    - if baseline is -1, after 2pos, how many becomes 0 or 1?
- counts of 1s for bridge

additionally for quality
- counts of 1s for repetition
- average fluency

In [41]:
# all files and dirs
baseline_path = "/scratch/fmeng/ActAdd/results/gemini_base/"
baseline_llama = "gemini_base_llama_senti+_fl_temp_0.json"
baseline_opt = "gemini_base_opt_senti+_fl_temp_0.json"
baseline_de = "gemini_base_de_senti+_fl.json"
baseline_zh = "gemini_base_zh_senti+_fl.json"

result_path = "/scratch/fmeng/ActAdd/results/"
dirs_llama = [
    "gemini_2pos_llama_senti+_fl_temp_0_no_space_hpt", 
    "gemini_2neg_llama_senti+_fl_temp_0_no_space_hpt", 
    "gemini_sent_2pos_llama_senti+_fl_temp_0_hpt", 
    "gemini_sent_2neg_llama_senti+_fl_temp_0_hpt"
    ]

dirs_opt = [
    "gemini_2pos_opt_senti+_fl_temp_0_no_space_hpt", 
    "gemini_2neg_opt_senti+_fl_temp_0_no_space_hpt", 
    "gemini_sent_2pos_opt_senti+_fl_temp_0_hpt",
    "gemini_sent_2neg_opt_senti+_fl_temp_0_hpt"
    ]

dirs_de = [
    "gemini_Love_de_senti+_fl",
    "gemini_Hate_de_senti+_fl",
    "gemini__love_de_senti+_fl",
    "gemini__hate_de_senti+_fl",
    "gemini_sent_2pos_de_senti+_fl",
    "gemini_sent_2neg_de_senti+_fl"
]

dirs_zh = [
    "gemini_Love_zh_senti+_fl",
    "gemini_Hate_zh_senti+_fl",
    "gemini__love_zh_senti+_fl",
    "gemini__hate_zh_senti+_fl",
    "gemini_sent_2pos_zh_senti+_fl",
    "gemini_sent_2neg_zh_senti+_fl"
]

dirs_bridge = [
    "gemini_bridge_llama_bridge+_fl_hpt", 
    "gemini_bridge_opt_bridge+_fl_hpt",
    "gemini_bridge_de_bridge+_fl",
    "gemini_bridge_zh_bridge+_fl"
    ]

# baseline
no generated sentence in the baseline is talking about the golden gate bridge

In [42]:
# baseline files have a different structure
def base_stats(dir, base_file):
    """
    return the sentiment labels of the base generation for downstream process
    """
    print("analysing ", base_file)
    df = pd.read_json(dir + base_file)
    print("counts of", df["continuation_label"].value_counts())
    print("number of repetitive sentences:", df["repetition"].sum().item())
    print("average perplexity of continuations:", df["fluency"].mean().item())
    print()
    return df["continuation_label"]

base_llama_sentimap = base_stats(baseline_path, baseline_llama)
base_opt_sentimap = base_stats(baseline_path, baseline_opt)
base_de_sentimap = base_stats(baseline_path, baseline_de)
base_zh_sentimap = base_stats(baseline_path, baseline_zh)

analysing  gemini_base_llama_senti+_fl_temp_0.json
counts of continuation_label
 0    13
 1     5
-1     2
Name: count, dtype: int64
number of repetitive sentences: 5
average perplexity of continuations: 2.557923251390457

analysing  gemini_base_opt_senti+_fl_temp_0.json
counts of continuation_label
 0    9
 1    7
-1    4
Name: count, dtype: int64
number of repetitive sentences: 12
average perplexity of continuations: 3.263213074207306

analysing  gemini_base_de_senti+_fl.json
counts of continuation_label
0    16
1     4
Name: count, dtype: int64
number of repetitive sentences: 11
average perplexity of continuations: 2.575465887784958

analysing  gemini_base_zh_senti+_fl.json
counts of continuation_label
 0    15
 1     3
-1     2
Name: count, dtype: int64
number of repetitive sentences: 7
average perplexity of continuations: 6.066083538532257



# sentiment
temperature 0
## harmonic mean

In [80]:
def dfs2hms(dfs):
    """
    harmonic mean
    """
    n = len(dfs)
    dfs_rec = list()
    for df in dfs:
        dfs_rec.append(1/df)
    sum_dfs_rec = sum(dfs_rec)
    hms = n/sum_dfs_rec
    return hms

def dfs2ams(dfs):
    """
    arithmetic mean
    """
    n = len(dfs)
    sum_dfs = sum(dfs)
    return sum_dfs/n

def get_means(success, repetition, fluency=None, mean=dfs2hms):
    """
    success: grid containing the counts of 1s or -1s, or increase relative to the baseline results
    repetition: 
    fluency: for opt results can be ignored
    https://www.datacamp.com/tutorial/how-to-normalize-data
    """
    # normalise success
    # https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.clip.html
    success = (success/20).clip(0.00001, 1)
    # print("success", success)
    # normalise repetion
    diversity = ((20-repetition)/20).clip(0.00001, 1)
    # print("diversity", diversity)
    if fluency is not None: 
        fl_max = fluency.max().max()  # 315.0967122733593
        fl_min = fluency.min().min()  # 1.8930507719516754
        numberator = fl_max - fluency
        denominator = fl_max - fl_min
        fl_norm = (numberator/denominator).clip(0.00001, 1)
        # print("fluency", fl_norm)
        print("fluency counted")
        hms = mean([success, diversity, fl_norm])
    else:
        print("fluency ignored")
        hms = mean([success, diversity])
    hms = hms.apply(pd.to_numeric).astype(float)
    return hms

## counting 1s or -1s
counting the number of positive/negative

In [ ]:
# process files in the whole directory, counting 
def senti_stats(dir, include_fl=False):
    print("showing result for directory", dir)
    lst_file = os.listdir(f"{result_path}{dir}/")
    n_files = len(lst_file)
    grid_one = pd.DataFrame(0, index=range(n_files),columns=range(20))
    grid_zero = pd.DataFrame(0, index=range(n_files),columns=range(20))
    grid_neg = pd.DataFrame(0, index=range(n_files),columns=range(20))
    grid_rep = pd.DataFrame(0, index=range(n_files),columns=range(20))
    grid_fl = pd.DataFrame(index=range(n_files),columns=range(20))
    for file_name in lst_file:
        layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
        with open(f"{result_path}{dir}/{file_name}", "r") as f: 
            r_dict = json.load(f)
        for coeff in r_dict:
            list_dict = pd.DataFrame(r_dict[coeff])
            coeff = int(coeff)
            if 1 in list_dict["continuation_label"].value_counts():
                grid_one.loc[layer, coeff-1] = list_dict["continuation_label"].value_counts()[1]
            if 0 in list_dict["continuation_label"].value_counts():
                grid_zero.loc[layer, coeff-1] = list_dict["continuation_label"].value_counts()[0]
            if -1 in list_dict["continuation_label"].value_counts():
                grid_neg.loc[layer, coeff-1] = list_dict["continuation_label"].value_counts()[-1]
            grid_rep.loc[layer, coeff-1] = list_dict["repetition"].sum().item()
            grid_fl.loc[layer, coeff-1] = list_dict["fluency"].mean().item()
    # https://stackoverflow.com/questions/12286607/making-heatmap-from-pandas-dataframe
    # https://stackoverflow.com/questions/61363712/how-to-print-a-pandas-io-formats-style-styler-object
    print("count of repetitive sentences ↓")
    display(grid_rep.style.background_gradient(cmap='Reds', axis=None))
    print("average perplexity of continuations ↓")
    gmap_clipped = grid_fl.clip(upper=grid_fl.quantile(0.95), axis=1)
    display(
        grid_fl.style.background_gradient(cmap='Reds', gmap=gmap_clipped, axis=None).format("{:,.2f}")
    )
    if not include_fl:
        grid_fl = None
    if "_2pos" in dir or "__love" in dir or "_Love" in dir: 
        print("count of positive continuation ↑")
        display(grid_one.style.background_gradient(cmap='Blues', axis=None))
        print("harmonic mean ↑")
        hms = get_means(grid_one, grid_rep, grid_fl)
        display(hms.style.background_gradient(cmap='Blues', axis=None))
    if "_2neg" in dir or "_Hate" in dir or "__hate" in dir:
        print("count of negative continuation ↑")
        display(grid_neg.style.background_gradient(cmap='Blues', axis=None))
        print("harmonic mean ↑")
        hms = get_means(grid_neg, grid_rep, grid_fl)
        # hms = hms.apply(pd.to_numeric).astype(float)
        display(hms.style.background_gradient(cmap='Blues', axis=None))


In [48]:
for dir in dirs_de:
    senti_stats(dir)

showing result for directory gemini_Love_de_senti+_fl
count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,11,9,11,8,12,16,14,15,18,15,17,16,17,17,19,19,18,18,17,18
1,7,14,19,14,17,19,20,20,20,20,20,20,20,19,20,20,20,20,20,20
2,9,10,13,17,19,17,18,20,20,20,20,20,20,20,20,20,20,20,20,20
3,9,11,14,12,18,18,16,18,17,19,20,20,20,20,20,20,19,20,20,19
4,11,13,15,17,19,18,20,18,17,18,18,18,19,19,20,20,19,20,20,19
5,9,8,13,14,15,15,18,15,19,17,20,20,20,20,19,20,18,18,19,19
6,9,7,13,13,12,14,14,14,17,18,17,19,17,18,19,20,18,18,20,20
7,12,11,9,13,9,16,18,15,18,19,19,18,18,20,20,19,20,20,19,19
8,13,10,11,11,11,14,14,15,16,15,18,17,18,19,19,19,18,18,19,20
9,12,12,12,8,12,12,15,16,17,18,18,17,18,19,19,20,20,19,20,20


average perplexity of continuations ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2.60,2.47,2.72,2.59,2.42,2.31,2.54,2.39,2.41,2.60,2.47,2.65,2.85,3.19,2.81,2.63,2.62,2.84,2.68,2.67
1,2.83,2.35,2.31,11.97,145.72,143.79,1.65,1.67,1.66,1.68,1.72,1.73,1.87,5.78,1.90,2.02,2.00,2.01,2.11,2.14
2,2.60,2.43,2.44,2.50,2.07,7.09,3.45,4.49,2.40,2.29,7.16,2.03,1.99,1.81,1.89,1.88,1.78,1.85,1.80,1.81
3,2.58,2.44,2.41,2.51,2.46,2.78,3.88,4.82,5.53,3.83,3.45,3.37,2.75,2.74,2.66,2.22,268.74,2.80,2.60,4.40
4,2.48,2.55,2.37,2.39,2.46,2.44,2.23,2.34,"184,004.31",2.29,2.37,2.43,1.94,1.89,1.79,1.80,1.76,1.70,1.70,1.75
5,2.43,2.61,2.35,2.25,2.39,2.53,2.56,2.78,3.16,3.41,3.27,2.67,2.98,2.92,3.19,2.71,2.85,2.62,2.48,2.28
6,2.57,2.67,2.52,2.42,2.39,2.61,2.67,2.91,2.93,2.95,3.04,3.28,3.45,3.34,3.44,3.78,3.28,3.86,4.15,3.50
7,2.54,2.42,2.44,2.37,2.43,2.50,2.51,2.83,3.38,3.40,3.40,3.52,3.89,3.78,3.08,2.96,3.21,3.40,2.91,2.97
8,2.59,2.65,2.63,2.48,2.63,2.62,2.49,2.72,2.87,3.04,2.97,2.92,2.91,3.00,3.48,3.63,3.73,3.78,4.02,3.76
9,2.61,2.51,2.63,2.82,2.54,2.51,2.64,2.61,2.89,3.14,3.83,3.48,4.16,3.71,4.00,4.55,3.95,5.07,4.26,4.51


count of positive continuation ↑


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2.56,2.55,2.44,2.53,2.58,2.57,2.52,2.47,2.47,2.39,2.40,2.48,2.45,2.43,2.40,2.34,2.41,2.47,2.47,2.46
1,2.61,2.58,2.58,2.45,2.58,2.54,2.47,2.36,2.52,2.45,2.43,2.47,2.50,2.51,2.45,2.31,2.28,2.28,2.33,2.26
2,2.46,2.59,2.47,2.72,2.60,2.45,2.48,2.42,2.49,2.43,2.24,2.28,2.30,2.32,2.29,2.22,2.20,2.30,2.25,2.11
3,2.57,2.69,2.61,2.69,2.52,2.42,2.51,2.42,2.31,2.26,2.35,2.45,2.34,2.40,2.29,2.30,2.36,2.66,2.49,2.51
4,2.43,2.69,2.55,4.21,4.26,6.24,4.23,4.29,3.40,16.37,10.17,4.11,3.91,5.65,7.16,7.52,6.13,3.64,4.69,10.80
5,2.59,2.49,2.41,2.37,2.39,2.45,2.45,2.50,2.63,2.61,2.70,2.64,2.39,2.44,2.52,2.36,2.47,2.51,2.60,2.63
6,2.71,2.57,2.50,2.43,2.46,2.32,2.24,2.21,2.43,2.51,2.70,2.59,2.59,2.43,2.35,2.30,2.45,2.42,2.58,2.69
7,2.61,2.46,2.36,2.44,2.46,2.63,3.38,2.87,3.53,17.99,21.60,23.13,36.16,33.58,27.60,37.25,34.74,56.58,42.25,46.96
8,2.45,2.43,2.51,2.35,2.37,2.28,2.50,2.68,2.59,2.54,2.61,2.53,2.64,2.66,2.57,2.60,2.81,2.76,2.62,2.73
9,2.42,2.31,2.46,2.43,2.41,2.46,2.48,2.48,2.64,2.64,2.64,2.55,2.51,2.59,2.43,2.58,2.61,2.45,2.67,2.52


count of negative continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,2,1,1,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1
4,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,1,0,1
5,0,0,0,0,0,0,0,0,0,1,1,1,0,0,2,1,0,0,0,0
6,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1
7,0,0,0,0,0,0,0,0,0,1,1,1,4,3,5,3,0,3,3,0
8,0,0,0,0,1,0,0,0,0,1,3,3,3,2,2,4,3,3,7,7
9,0,0,0,0,1,0,0,0,0,1,2,0,0,1,1,1,2,2,3,3


harmonic mean ↑
fluency ignored


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.000020,0.088889,0.000020,0.000020,0.000020,0.092308,0.000020,0.090909,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020
1,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.088889,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.085714,0.085714,0.000020,0.000020,0.000020,0.000020,0.000020
2,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.087500,0.133333,0.083333,0.083333,0.000020,0.000020
3,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.085714,0.083333
4,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.083333,0.080000,0.083333,0.085714,0.000020,0.000020,0.087500,0.000020,0.085714
5,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.087500,0.088889,0.090909,0.000020,0.000020,0.160000,0.091667,0.000020,0.000020,0.000020,0.000020
6,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.083333,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.090000,0.090000
7,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.066667,0.066667,0.050000,0.133333,0.075000,0.000020,0.000020,0.000010,0.000020,0.000020,0.000010
8,0.000020,0.000020,0.000020,0.000020,0.085714,0.000020,0.000020,0.000020,0.000020,0.090909,0.225000,0.225000,0.210000,0.163636,0.155556,0.222222,0.171429,0.150000,0.291667,0.210000
9,0.000020,0.000020,0.000020,0.000020,0.088889,0.000020,0.000020,0.000020,0.000020,0.090909,0.160000,0.000020,0.000020,0.085714,0.083333,0.083333,0.155556,0.120000,0.200000,0.187500


## comparing with baseline

In [ ]:
# base_llama_sentimap[10].item()  # 0, 1, -1
def comparative_stats(dir, sentimap, include_fl=False):
    """
    base_llama_sentimap or base_opt_sentimap 
    gemini_2pos_llama_senti+_fl_temp_0_no_space_hpt
    """
    print("showing result for directory", dir)    
    lst_files = os.listdir(f"{result_path}{dir}/")
    n_files = len(lst_files)
    grid_success = pd.DataFrame(0, index=range(n_files),columns=range(20))
    grid_rep = pd.DataFrame(0, index=range(n_files),columns=range(20))
    grid_fl = pd.DataFrame(index=range(n_files),columns=range(20))
    if "_2pos" in dir or "__love" in dir or "_Love" in dir:  # count the tags that are larger than the corresponding one in the base_map
        for file_name in lst_files:
            layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
            with open(f"{result_path}{dir}/{file_name}", "r") as f: 
                r_dict = json.load(f)
            for coeff in r_dict:
                list_dict = pd.DataFrame(r_dict[coeff])
                coeff = int(coeff)
                # pointwise compare with llama_sentimap
                successs = list_dict["continuation_label"] > sentimap
                grid_success.loc[layer, coeff-1] = successs.sum()
                grid_rep.loc[layer, coeff-1] = list_dict["repetition"].sum().item()
                grid_fl.loc[layer, coeff-1] = list_dict["fluency"].mean().item()
    if "_2neg" in dir or "_Hate" in dir or "__hate" in dir:  # count the tags that are smaller than the corresponding one in the base_map
        for file_name in lst_files:
            layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
            with open(f"{result_path}{dir}/{file_name}", "r") as f: 
                r_dict = json.load(f)
            for coeff in r_dict:
                list_dict = pd.DataFrame(r_dict[coeff])
                coeff = int(coeff)
                successs = list_dict["continuation_label"] < sentimap
                grid_success.loc[layer, coeff-1] = successs.sum()
                grid_rep.loc[layer, coeff-1] = list_dict["repetition"].sum().item()
                grid_fl.loc[layer, coeff-1] = list_dict["fluency"].mean().item()
    print("count of bridges ↑")
    display(grid_success.style.background_gradient(cmap='Blues', axis=None))
    if not include_fl:
        grid_fl = None
    print("harmonic mean ↑")
    hms = get_means(grid_success, grid_rep, grid_fl)
    # hms = hms.apply(pd.to_numeric).astype(float)
    display(hms.style.background_gradient(cmap='Blues', axis=None))


In [53]:
for dir in dirs_de:
    comparative_stats(dir, base_de_sentimap)

showing result for directory gemini_Love_de_senti+_fl
count of bridges ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5,7,9,8,4,4,8,4,8,7,5,7,7,8,9,6,8,7,6,6
1,4,5,6,2,0,1,1,0,1,0,1,1,3,2,3,1,2,1,1,0
2,8,7,8,7,7,8,4,5,5,0,4,6,5,4,5,3,7,8,7,8
3,6,6,5,5,8,10,7,4,6,5,7,6,8,9,6,6,5,8,6,5
4,3,3,6,5,3,5,8,8,8,8,9,7,8,5,5,5,5,5,6,7
5,5,2,3,3,3,5,5,6,6,5,6,9,6,6,6,5,3,2,5,2
6,4,7,3,7,5,6,7,7,7,7,7,8,9,8,6,7,9,7,6,6
7,4,4,2,6,3,4,6,8,5,6,5,8,5,4,4,3,1,3,5,5
8,2,3,5,3,6,7,6,7,8,5,6,3,5,6,5,5,7,6,6,5
9,3,2,7,2,5,7,7,6,6,9,5,2,1,2,3,4,5,3,1,2


harmonic mean ↑
fluency ignored


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.321429,0.427778,0.450000,0.480000,0.266667,0.200000,0.342857,0.222222,0.160000,0.291667,0.187500,0.254545,0.210000,0.218182,0.090000,0.085714,0.160000,0.155556,0.200000,0.150000
1,0.305882,0.272727,0.085714,0.150000,0.000020,0.050000,0.000020,0.000010,0.000020,0.000010,0.000020,0.000020,0.000020,0.066667,0.000020,0.000020,0.000020,0.000020,0.000020,0.000010
2,0.463158,0.411765,0.373333,0.210000,0.087500,0.218182,0.133333,0.000020,0.000020,0.000010,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020
3,0.388235,0.360000,0.272727,0.307692,0.160000,0.166667,0.254545,0.133333,0.200000,0.083333,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.083333,0.000020,0.000020,0.083333
4,0.225000,0.210000,0.272727,0.187500,0.075000,0.142857,0.000020,0.160000,0.218182,0.160000,0.163636,0.155556,0.088889,0.083333,0.000020,0.000020,0.083333,0.000020,0.000020,0.087500
5,0.343750,0.171429,0.210000,0.200000,0.187500,0.250000,0.142857,0.272727,0.085714,0.187500,0.000020,0.000020,0.000020,0.000020,0.085714,0.000020,0.120000,0.100000,0.083333,0.066667
6,0.293333,0.455000,0.210000,0.350000,0.307692,0.300000,0.323077,0.323077,0.210000,0.155556,0.210000,0.088889,0.225000,0.160000,0.085714,0.000020,0.163636,0.155556,0.000020,0.000020
7,0.266667,0.276923,0.169231,0.323077,0.235714,0.200000,0.150000,0.307692,0.142857,0.085714,0.083333,0.160000,0.142857,0.000020,0.000020,0.075000,0.000020,0.000020,0.083333,0.083333
8,0.155556,0.230769,0.321429,0.225000,0.360000,0.323077,0.300000,0.291667,0.266667,0.250000,0.150000,0.150000,0.142857,0.085714,0.083333,0.083333,0.155556,0.150000,0.085714,0.000020
9,0.218182,0.160000,0.373333,0.171429,0.307692,0.373333,0.291667,0.240000,0.200000,0.163636,0.142857,0.120000,0.066667,0.066667,0.075000,0.000020,0.000020,0.075000,0.000020,0.000020


showing result for directory gemini_Hate_de_senti+_fl
count of bridges ↑


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.150000,0.225000,0.350000,0.166667,0.218182,0.210000,0.088889,0.160000,0.225000,0.276923,0.293333,0.240000,0.293333,0.247059,0.243750,0.173333,0.090909,0.090000,0.091667,0.090909
1,0.210000,0.155556,0.150000,0.155556,0.291667,0.225000,0.225000,0.333333,0.276923,0.307692,0.321429,0.266667,0.200000,0.222222,0.222222,0.250000,0.307692,0.323077,0.360000,0.307692
2,0.218182,0.235714,0.333333,0.276923,0.266667,0.266667,0.085714,0.000020,0.120000,0.050000,0.000010,0.050000,0.050000,0.000020,0.000020,0.000020,0.000020,0.000020,0.050000,0.000010
3,0.155556,0.155556,0.187500,0.266667,0.160000,0.225000,0.285714,0.230769,0.321429,0.360000,0.307692,0.218182,0.200000,0.155556,0.160000,0.218182,0.350000,0.307692,0.321429,0.360000
4,0.166667,0.285714,0.291667,0.323077,0.411765,0.400000,0.291667,0.291667,0.350000,0.323077,0.323077,0.323077,0.350000,0.272727,0.291667,0.240000,0.150000,0.155556,0.155556,0.142857
5,0.085714,0.092308,0.187500,0.291667,0.210000,0.323077,0.291667,0.142857,0.222222,0.187500,0.120000,0.000020,0.000020,0.120000,0.066667,0.000020,0.050000,0.066667,0.075000,0.000020
6,0.169231,0.225000,0.300000,0.323077,0.375000,0.350000,0.307692,0.360000,0.342857,0.291667,0.307692,0.393750,0.360000,0.423529,0.373333,0.254545,0.210000,0.155556,0.142857,0.142857
7,0.276923,0.171429,0.321429,0.266667,0.321429,0.240000,0.200000,0.187500,0.171429,0.250000,0.210000,0.200000,0.276923,0.240000,0.375000,0.085714,0.083333,0.083333,0.133333,0.171429
8,0.240000,0.321429,0.266667,0.368421,0.305882,0.305882,0.300000,0.352941,0.243750,0.350000,0.272727,0.323077,0.321429,0.342857,0.266667,0.160000,0.187500,0.200000,0.187500,0.200000
9,0.166667,0.163636,0.243750,0.230769,0.166667,0.285714,0.225000,0.360000,0.300000,0.393750,0.393750,0.240000,0.266667,0.222222,0.200000,0.222222,0.254545,0.300000,0.323077,0.291667


showing result for directory gemini_sent_2neg_de_senti+_fl
count of bridges ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,1,3,3,0,2,2,1,2,2,1,1,1,2,2,2,2,2,2,1,2
1,0,0,0,3,0,4,3,2,3,3,3,3,2,2,3,1,2,2,2,2
2,1,1,2,1,2,1,2,2,1,1,1,1,1,1,2,3,1,2,2,1
3,1,1,1,1,0,1,2,1,1,1,2,2,3,3,2,2,3,3,2,2
4,0,1,0,2,2,2,1,2,2,3,1,3,3,3,3,3,2,4,3,3
5,0,0,1,1,1,0,1,1,2,1,1,2,2,2,2,3,2,2,1,1
6,0,1,1,2,1,1,1,2,1,2,3,3,3,2,2,1,1,1,4,3
7,0,0,2,1,2,2,2,2,3,4,4,2,6,5,6,5,2,7,6,4
8,2,2,2,3,4,3,3,3,3,2,4,5,5,4,3,4,4,5,9,9
9,2,2,1,2,3,2,1,2,1,2,2,0,0,1,2,2,3,3,5,5


harmonic mean ↑
fluency ignored


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.090000,0.218182,0.210000,0.000020,0.166667,0.171429,0.092857,0.166667,0.166667,0.090000,0.088889,0.088889,0.163636,0.160000,0.163636,0.155556,0.166667,0.166667,0.090909,0.163636
1,0.000020,0.000020,0.000020,0.218182,0.000020,0.285714,0.218182,0.160000,0.200000,0.210000,0.225000,0.235714,0.150000,0.150000,0.200000,0.087500,0.150000,0.150000,0.160000,0.160000
2,0.090000,0.090000,0.166667,0.092308,0.166667,0.090909,0.163636,0.160000,0.087500,0.087500,0.075000,0.080000,0.085714,0.085714,0.155556,0.171429,0.083333,0.142857,0.120000,0.075000
3,0.090000,0.091667,0.091667,0.092308,0.000020,0.090909,0.163636,0.087500,0.085714,0.080000,0.150000,0.155556,0.187500,0.187500,0.142857,0.133333,0.200000,0.225000,0.150000,0.142857
4,0.000020,0.090909,0.000020,0.160000,0.160000,0.155556,0.090000,0.155556,0.160000,0.218182,0.090000,0.187500,0.171429,0.187500,0.200000,0.210000,0.150000,0.254545,0.210000,0.200000
5,0.000020,0.000020,0.087500,0.085714,0.085714,0.000020,0.088889,0.087500,0.155556,0.087500,0.088889,0.166667,0.155556,0.160000,0.160000,0.235714,0.171429,0.166667,0.090000,0.090000
6,0.000020,0.090909,0.088889,0.166667,0.088889,0.088889,0.085714,0.142857,0.083333,0.150000,0.200000,0.200000,0.210000,0.150000,0.133333,0.080000,0.083333,0.080000,0.276923,0.225000
7,0.000020,0.000020,0.150000,0.066667,0.120000,0.155556,0.163636,0.150000,0.171429,0.133333,0.133333,0.066667,0.150000,0.083333,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020
8,0.160000,0.163636,0.150000,0.200000,0.240000,0.187500,0.210000,0.230769,0.235714,0.166667,0.276923,0.321429,0.291667,0.276923,0.210000,0.222222,0.200000,0.187500,0.321429,0.225000
9,0.163636,0.163636,0.091667,0.163636,0.218182,0.169231,0.091667,0.160000,0.091667,0.166667,0.160000,0.000020,0.000020,0.085714,0.142857,0.142857,0.210000,0.150000,0.272727,0.250000


# talking about the bridge

In [ ]:
def bridge_stats(dir, include_fl=False):
    """
    counting the number of steered sentences that talk about the golden gate bridge 
    """
    print("showing result for directory", dir)
    lst_files = os.listdir(f"{result_path}{dir}/")
    n_file = len(lst_files)
    grid_bridge = pd.DataFrame(0, index=range(n_file),columns=range(20))
    grid_rep = pd.DataFrame(0, index=range(n_file),columns=range(20))
    grid_fl = pd.DataFrame(index=range(n_file),columns=range(20))
    for file_name in lst_files:
        layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
        with open(f"{result_path}{dir}/{file_name}", "r") as f: 
            r_dict = json.load(f)
        for coeff in r_dict:
            list_dict = pd.DataFrame(r_dict[coeff])
            coeff = int(coeff)
            if 1 in list_dict["bridge"].value_counts():
                grid_bridge.loc[layer, coeff-1] = list_dict["bridge"].value_counts()[1]
            else:
                grid_bridge.loc[layer, coeff-1] = 0
            grid_rep.loc[layer, coeff-1] = list_dict["repetition"].sum().item()
            grid_fl.loc[layer, coeff-1] = list_dict["fluency"].mean().item()
    # https://stackoverflow.com/questions/12286607/making-heatmap-from-pandas-dataframe
    # https://stackoverflow.com/questions/61363712/how-to-print-a-pandas-io-formats-style-styler-object

    print("count of repetitive sentences ↓")
    display(grid_rep.style.background_gradient(cmap='Reds', axis=None))
    print("average perplexity of continuations ↓")
    gmap_clipped = grid_fl.clip(upper=grid_fl.quantile(0.95), axis=1)
    display(
        grid_fl.style.background_gradient(cmap='Reds', gmap=gmap_clipped, axis=None).format("{:,.2f}")
    )
    print("count of sentences talking about the bridge ↑")
    display(grid_bridge.style.background_gradient(cmap='Blues', axis=None))
    if not include_fl:
        grid_fl = None
    print("harmonic mean ↑")
    hms = get_means(grid_bridge, grid_rep, grid_fl)
    display(hms.style.background_gradient(cmap='Blues', axis=None))


In [56]:
for dir in dirs_bridge:
    bridge_stats(dir)

showing result for directory gemini_bridge_llama_bridge+_fl_hpt
count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5,11,14,18,18,19,20,20,20,20,20,19,20,20,20,20,20,20,20,20
1,7,4,14,16,19,19,20,19,20,18,20,20,20,20,20,20,20,20,20,20
2,8,6,8,13,12,11,10,11,10,11,11,14,12,18,19,20,20,20,20,20
3,10,7,10,8,11,8,6,5,6,10,3,7,10,16,20,19,20,20,20,19
4,7,7,10,10,10,13,8,12,18,20,19,20,19,19,20,20,20,20,20,20
5,8,10,9,7,15,13,13,15,19,20,20,20,19,19,19,19,20,20,20,20
6,5,11,8,8,4,9,14,14,16,17,16,17,19,19,18,18,18,19,19,19
7,9,10,10,9,13,14,16,16,14,17,18,19,19,18,19,20,20,20,20,19
8,5,7,10,4,7,11,12,15,18,17,16,18,18,19,20,20,20,19,19,19
9,10,6,8,11,8,9,9,11,14,17,18,18,18,18,19,19,19,18,17,18


average perplexity of continuations ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2.88,5.55,8.75,8.04,9.28,13.83,14.68,15.72,11.76,12.00,10.84,8.53,11.51,9.49,26.63,76.70,32.44,10.12,60.70,12.53
1,2.84,2.93,3.04,3.10,2.78,2.54,2.66,2.97,2.83,2.83,2.61,2.42,2.47,2.50,2.46,2.43,2.41,2.24,2.22,2.20
2,2.72,2.92,3.85,3.18,2.91,3.42,3.56,4.47,6.76,8.44,8.82,7.70,6.94,4.30,3.47,3.80,4.25,4.63,3.06,3.71
3,2.78,3.40,3.02,3.63,3.33,3.93,4.41,4.98,4.85,4.76,6.30,5.58,5.40,4.54,4.23,3.45,3.50,3.95,3.54,3.75
4,2.70,3.06,3.12,3.20,3.30,3.56,3.26,2.89,2.67,2.52,2.61,2.81,2.79,2.80,2.76,2.84,2.86,2.61,2.75,2.65
5,2.83,2.81,3.00,2.93,3.09,2.80,2.89,2.76,3.25,3.22,3.50,3.60,3.57,3.85,3.72,3.97,4.22,3.84,3.30,3.46
6,2.82,2.84,2.92,3.24,3.33,3.12,2.99,3.74,3.47,3.48,3.59,3.51,3.11,3.52,3.18,2.95,3.05,3.17,2.90,3.03
7,2.75,2.62,2.57,2.72,2.63,2.73,3.03,3.13,3.19,2.98,3.02,3.04,3.01,2.93,3.02,2.93,3.05,3.18,3.02,3.01
8,2.75,2.89,2.79,2.85,2.84,3.17,2.96,2.98,3.16,3.56,3.58,3.69,3.99,4.00,3.64,3.86,4.19,3.67,3.69,3.63
9,2.48,3.32,2.92,3.01,3.07,3.13,3.51,3.46,2.96,3.14,3.42,3.56,3.72,3.38,3.46,3.45,3.73,3.79,3.80,4.45


count of sentences talking about the bridge ↑


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5.39,5.18,6.42,9.66,"1,089.37",30.92,"4,578.84","1,860,835.88","15,646.35","38,598.51",875.09,"1,833.84",110.22,560.45,811.08,798.00,807.16,"8,631.08","263,048.87","330,623.44"
1,4.89,5.69,6.31,5.45,4.84,5.04,3.60,3.13,3.31,3.20,2.57,2.65,2.55,2.60,2.82,2.69,2.69,2.97,3.32,3.61
2,5.59,4.66,5.73,4.74,4.38,4.60,3.74,3.02,3.18,3.74,4.43,3.61,2.47,2.42,2.46,2.87,3.27,2.89,3.17,3.62
3,6.08,5.73,4.92,5.28,6.99,21.79,22.12,16.79,32.51,14.35,8.68,5.15,6.17,4.62,3.64,3.90,4.32,4.90,4.97,5.98
4,5.51,4.77,4.36,4.88,5.33,5.27,4.32,5.38,5.37,4.70,4.94,5.86,4.97,66.90,69.12,67.56,70.34,71.52,70.00,69.60
5,5.33,5.02,4.58,4.69,4.87,4.67,5.33,5.73,6.26,5.50,4.65,6.16,7.02,16.97,15.85,112.84,113.59,116.22,115.75,115.18
6,4.89,5.45,4.81,4.26,4.11,4.76,4.27,4.97,6.01,8.37,10.22,12.94,14.45,214.34,219.21,212.04,215.66,209.90,206.00,274.92
7,5.72,6.05,5.40,5.53,5.23,5.44,5.32,6.21,10.41,10.80,19.42,18.81,20.80,14.19,15.48,11.60,12.74,14.09,13.56,20.32
8,5.87,5.83,5.89,5.10,5.99,7.25,6.86,10.26,12.25,14.58,14.50,14.63,14.42,15.05,15.34,10.29,12.41,12.09,12.12,13.06
9,5.73,5.30,5.65,5.19,5.68,5.76,8.59,7.90,7.60,10.24,17.70,12.31,12.90,10.14,15.08,14.81,17.66,23.62,23.79,13.34


count of sentences talking about the bridge ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,4,5,5,3,7,1,2,3,5,0,1,1,0,0,0,0,0,1,3,3
1,3,4,6,2,6,7,3,4,4,5,2,1,0,0,0,0,0,0,3,3
2,1,6,5,7,8,8,8,6,3,3,6,4,3,5,7,9,10,12,12,12
3,1,4,8,8,6,7,4,6,4,5,10,16,13,12,17,14,12,14,12,11
4,0,5,7,5,1,0,1,3,6,9,7,6,4,5,5,7,5,6,6,7
5,0,3,6,2,0,3,4,8,10,10,13,12,9,8,10,8,7,7,9,9
6,0,4,6,7,5,4,6,8,8,8,4,8,10,8,9,6,5,4,5,4
7,0,1,3,2,2,4,2,2,1,2,2,2,2,2,2,2,1,2,1,1
8,0,1,2,3,2,4,0,2,1,1,1,2,3,4,2,3,2,2,2,2
9,0,1,0,1,1,0,0,2,1,3,3,2,3,3,2,2,2,4,5,7


harmonic mean ↑
fluency ignored


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.266667,0.307692,0.291667,0.187500,0.000020,0.000020,0.066667,0.075000,0.000020,0.000020,0.000020,0.050000,0.000020,0.000020,0.000010,0.000010,0.000020,0.066667,0.120000,0.150000
1,0.187500,0.300000,0.300000,0.120000,0.085714,0.087500,0.000020,0.000020,0.000020,0.000020,0.000020,0.050000,0.000010,0.000010,0.000010,0.000010,0.000010,0.000020,0.000020,0.000020
2,0.090909,0.388235,0.272727,0.427778,0.342857,0.342857,0.266667,0.200000,0.120000,0.075000,0.085714,0.080000,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020
3,0.090000,0.305882,0.480000,0.444444,0.342857,0.393750,0.222222,0.240000,0.200000,0.222222,0.333333,0.320000,0.361111,0.092308,0.094444,0.000020,0.000020,0.000020,0.000020,0.000020
4,0.000020,0.352941,0.427778,0.291667,0.083333,0.000020,0.080000,0.075000,0.240000,0.225000,0.155556,0.272727,0.080000,0.187500,0.187500,0.210000,0.222222,0.240000,0.272727,0.291667
5,0.000020,0.210000,0.342857,0.166667,0.000020,0.210000,0.222222,0.266667,0.333333,0.230769,0.173333,0.300000,0.276923,0.218182,0.166667,0.218182,0.254545,0.291667,0.276923,0.276923
6,0.000020,0.293333,0.375000,0.427778,0.291667,0.305882,0.388235,0.342857,0.307692,0.373333,0.276923,0.266667,0.230769,0.373333,0.393750,0.300000,0.272727,0.240000,0.272727,0.254545
7,0.000020,0.085714,0.225000,0.175000,0.150000,0.254545,0.166667,0.171429,0.091667,0.171429,0.171429,0.171429,0.173333,0.169231,0.171429,0.160000,0.090000,0.169231,0.091667,0.093333
8,0.000020,0.092857,0.166667,0.218182,0.166667,0.293333,0.000020,0.166667,0.091667,0.092308,0.093333,0.173333,0.240000,0.300000,0.166667,0.225000,0.169231,0.163636,0.160000,0.155556
9,0.000020,0.090000,0.000020,0.090000,0.092308,0.000020,0.000020,0.160000,0.087500,0.230769,0.225000,0.171429,0.243750,0.240000,0.171429,0.173333,0.173333,0.300000,0.352941,0.373333


# de with Llama-3.1-8B

In [64]:
llama3_1_path = "/scratch/fmeng/ActAdd/results/de_llama3.1/"
baseline_llama3_1 = "gemini_base_de_fl_senti+_temp_0.json"
dirs_llama3_1 = [
    "de_llama3.1/gemini_2pos_de_fl_senti+_temp_0_no_space",
    "de_llama3.1/gemini_2neg_de_fl_senti+_temp_0_no_space",
    "de_llama3.1/gemini_sent_2pos_de_fl_senti+_temp_0",
    "de_llama3.1/gemini_sent_2neg_de_fl_senti+_temp_0"
]
dirs_bridge_llama3_1 = "de_llama3.1/gemini_bridge_de_fl_bridge+"

In [60]:
base_llama3_1_sentimap = base_stats(llama3_1_path, baseline_llama3_1)

analysing  gemini_base_de_fl_senti+_temp_0.json
counts of continuation_label
0    15
1     5
Name: count, dtype: int64
number of repetitive sentences: 13
average perplexity of continuations: 2.9126860082149504



## counting 1s or -1s

In [61]:
for dir in dirs_llama3_1:
    senti_stats(dir)

showing result for directory de_llama3.1/gemini_2pos_de_fl_senti+_temp_0_no_space
count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,14,12,14,12,12,10,11,12,11,12,8,10,13,12,12,12,14,14,13,13
1,11,13,18,20,19,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20
2,14,15,14,15,17,19,17,18,18,18,19,19,18,19,20,20,20,20,20,20
3,12,13,13,15,11,12,11,16,15,16,19,18,19,19,18,20,20,20,19,19
4,16,15,17,11,12,12,14,17,17,18,17,16,17,18,19,18,19,19,19,20
5,15,16,10,12,8,10,16,13,13,13,14,16,17,15,17,17,17,15,17,17
6,15,16,16,12,10,13,15,13,16,14,13,15,14,15,13,13,14,14,12,15
7,14,14,15,13,11,12,10,12,11,13,13,14,11,14,14,14,13,14,15,14
8,14,13,12,11,12,11,12,9,10,13,12,13,13,14,15,14,13,13,15,16
9,15,13,14,13,12,12,9,10,9,10,12,12,11,9,9,10,11,11,10,11


average perplexity of continuations ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2.83,2.93,3.15,2.87,2.99,3.08,2.99,2.83,2.82,2.81,3.00,2.86,2.99,2.93,2.86,2.81,2.67,2.79,2.90,3.04
1,3.03,3.04,2.69,2.79,8.88,1.80,1.95,1.94,2.01,2.02,2.04,1.98,1.94,1.91,1.92,2.04,1.99,2.08,2.14,2.20
2,2.93,2.95,3.01,3.03,2.77,2.98,3.48,5.72,4.32,6.49,10.87,7.60,383.82,6.61,4.89,7.00,6.09,3.45,6.43,4.69
3,3.03,2.94,2.82,2.89,3.03,3.22,3.29,3.66,4.34,4.43,5.88,6.12,6.16,6.81,5.61,6.84,7.73,7.01,15.49,8.07
4,2.97,2.89,2.72,2.84,3.00,3.09,2.94,2.70,2.77,2.89,3.03,3.00,3.10,3.51,3.49,3.47,3.56,3.50,3.39,3.17
5,2.90,2.88,3.11,3.10,3.01,3.04,3.04,2.98,2.99,3.21,3.42,3.61,3.42,3.48,3.31,3.33,3.54,3.73,3.94,3.86
6,2.94,2.98,2.94,3.03,3.02,3.10,3.21,3.22,3.22,3.27,3.23,3.29,3.66,3.72,3.76,3.77,3.85,3.90,3.97,4.07
7,2.86,3.01,2.79,2.84,2.88,2.84,3.03,3.19,3.26,3.55,3.60,3.58,3.76,4.02,4.14,4.06,4.50,4.97,4.76,4.91
8,2.90,3.00,2.93,2.89,3.12,2.82,2.90,2.92,2.92,2.94,3.06,3.00,3.06,3.12,3.37,3.39,3.68,3.73,3.85,3.78
9,2.96,2.94,2.94,2.92,3.30,3.18,3.32,3.41,3.31,3.31,3.25,3.38,3.33,3.48,3.44,3.44,3.62,3.60,3.71,3.64


count of positive continuation ↑


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2.91,2.90,3.03,2.93,2.79,2.79,2.78,2.78,2.80,2.74,2.84,2.81,2.82,2.90,2.95,2.95,2.93,2.96,3.02,3.01
1,2.87,2.88,2.99,2.86,2.86,2.80,2.78,2.86,2.77,2.84,2.79,2.84,2.82,2.73,2.71,2.80,2.76,2.80,2.78,2.78
2,2.93,2.94,2.97,2.93,3.08,2.94,2.89,2.87,2.92,3.01,2.78,2.72,2.75,2.90,2.92,2.81,2.74,2.65,2.72,2.77
3,2.91,2.92,2.82,2.94,2.83,2.87,2.73,2.71,2.66,2.59,2.57,2.70,2.69,2.78,2.77,2.75,2.66,2.76,2.77,2.75
4,2.89,3.03,3.24,4.04,6.60,3.49,3.48,7.23,5.09,6.01,4.03,7.12,82.54,12.49,6.39,4.41,9.70,8.20,11.65,17.08
5,2.87,3.07,2.98,2.82,2.84,2.81,2.83,2.86,2.85,2.88,2.90,2.90,2.91,2.89,3.00,3.04,3.15,3.15,2.99,2.95
6,2.87,2.98,2.81,2.79,2.82,2.79,2.83,2.89,2.98,2.99,2.95,2.87,2.88,2.96,2.91,3.08,3.02,2.97,2.95,3.09
7,2.89,2.86,2.87,2.77,2.78,2.94,3.01,2.94,2.93,2.93,2.96,2.88,2.91,2.87,2.95,3.04,3.37,3.57,3.75,4.49
8,2.91,2.91,2.83,2.91,3.00,3.09,2.99,3.07,3.25,3.26,3.36,3.32,3.39,3.23,3.23,3.12,3.18,3.21,3.29,2.96
9,2.90,2.90,2.79,2.85,2.77,2.89,2.93,2.88,2.99,3.05,3.06,3.16,3.27,3.23,3.15,3.02,3.07,3.15,3.13,3.03


count of negative continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0,1,0,0,0,0,0,1,1,1,2,2,1,1,1,2,2,2,2,2
1,0,1,0,1,1,2,1,1,1,1,0,1,0,0,0,0,0,0,1,1
2,0,0,1,1,1,2,1,0,0,0,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,1,2,2,1,1,0,0,2,1,0,1,0,0
5,0,1,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1
6,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,2,1,2,2,1,0,0,0,1,0,0,0,0,4,3,3,2
8,0,0,0,0,1,2,2,2,3,2,2,2,2,3,2,2,3,3,1,1
9,0,0,0,0,0,0,0,0,0,1,1,1,2,1,0,0,1,1,3,2


harmonic mean ↑
fluency ignored


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.000020,0.083333,0.000020,0.000020,0.000020,0.000020,0.000020,0.080000,0.075000,0.080000,0.150000,0.150000,0.088889,0.088889,0.087500,0.155556,0.160000,0.166667,0.166667,0.163636
1,0.000020,0.085714,0.000020,0.083333,0.088889,0.160000,0.088889,0.090000,0.090909,0.090000,0.000020,0.088889,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.090000,0.087500
2,0.000020,0.000020,0.090909,0.087500,0.090000,0.163636,0.088889,0.000020,0.000020,0.000020,0.075000,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020
3,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.087500,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020
4,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.090909,0.166667,0.160000,0.087500,0.075000,0.000020,0.000020,0.142857,0.083333,0.000020,0.085714,0.000020,0.000020
5,0.000020,0.080000,0.000020,0.087500,0.088889,0.087500,0.088889,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.085714
6,0.000020,0.080000,0.083333,0.000020,0.000020,0.000020,0.000020,0.000020,0.090000,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020
7,0.000020,0.000020,0.000020,0.150000,0.085714,0.142857,0.133333,0.088889,0.000020,0.000020,0.000020,0.080000,0.000020,0.000020,0.000020,0.000020,0.171429,0.150000,0.120000,0.133333
8,0.000020,0.000020,0.000020,0.000020,0.085714,0.150000,0.160000,0.155556,0.218182,0.160000,0.163636,0.160000,0.160000,0.218182,0.160000,0.142857,0.187500,0.171429,0.083333,0.080000
9,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.090909,0.088889,0.090909,0.163636,0.083333,0.000020,0.000020,0.087500,0.085714,0.187500,0.100000


## comparing with baseline

In [62]:
for dir in dirs_llama3_1:
    comparative_stats(dir, base_llama3_1_sentimap)

showing result for directory de_llama3.1/gemini_2pos_de_fl_senti+_temp_0_no_space
count of bridges ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,4,7,6,5,7,6,6,8,7,8,5,5,5,5,6,5,5,7,5,4
1,6,9,7,2,0,0,2,1,0,0,0,0,0,2,1,1,1,1,1,2
2,1,4,6,3,5,7,5,7,6,4,4,3,1,2,1,3,2,4,4,3
3,2,2,3,6,5,8,7,6,8,8,9,5,9,7,8,6,6,7,7,6
4,3,4,4,8,9,7,7,11,7,8,9,8,6,6,9,9,9,11,10,7
5,1,3,4,3,5,6,7,7,7,9,7,9,10,6,7,6,9,9,10,11
6,2,4,3,4,6,3,3,3,3,5,6,5,7,9,9,8,7,7,6,6
7,1,2,4,3,5,8,5,6,6,7,7,4,4,4,6,6,6,6,3,2
8,1,2,2,2,4,3,3,3,3,5,6,5,5,6,6,5,5,8,9,7
9,1,2,3,3,3,4,4,5,5,3,4,5,4,4,4,4,5,5,5,5


harmonic mean ↑
fluency ignored


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.240000,0.373333,0.300000,0.307692,0.373333,0.375000,0.360000,0.400000,0.393750,0.400000,0.352941,0.333333,0.291667,0.307692,0.342857,0.307692,0.272727,0.323077,0.291667,0.254545
1,0.360000,0.393750,0.155556,0.000020,0.000020,0.000010,0.000020,0.000020,0.000010,0.000010,0.000010,0.000010,0.000010,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020
2,0.085714,0.222222,0.300000,0.187500,0.187500,0.087500,0.187500,0.155556,0.150000,0.133333,0.080000,0.075000,0.066667,0.066667,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020
3,0.160000,0.155556,0.210000,0.272727,0.321429,0.400000,0.393750,0.240000,0.307692,0.266667,0.090000,0.142857,0.090000,0.087500,0.160000,0.000020,0.000020,0.000020,0.087500,0.085714
4,0.171429,0.222222,0.171429,0.423529,0.423529,0.373333,0.323077,0.235714,0.210000,0.160000,0.225000,0.266667,0.200000,0.150000,0.090000,0.163636,0.090000,0.091667,0.090909,0.000020
5,0.083333,0.171429,0.285714,0.218182,0.352941,0.375000,0.254545,0.350000,0.350000,0.393750,0.323077,0.276923,0.230769,0.272727,0.210000,0.200000,0.225000,0.321429,0.230769,0.235714
6,0.142857,0.200000,0.171429,0.266667,0.375000,0.210000,0.187500,0.210000,0.171429,0.272727,0.323077,0.250000,0.323077,0.321429,0.393750,0.373333,0.323077,0.323077,0.342857,0.272727
7,0.085714,0.150000,0.222222,0.210000,0.321429,0.400000,0.333333,0.342857,0.360000,0.350000,0.350000,0.240000,0.276923,0.240000,0.300000,0.300000,0.323077,0.300000,0.187500,0.150000
8,0.085714,0.155556,0.160000,0.163636,0.266667,0.225000,0.218182,0.235714,0.230769,0.291667,0.342857,0.291667,0.291667,0.300000,0.272727,0.272727,0.291667,0.373333,0.321429,0.254545
9,0.083333,0.155556,0.200000,0.210000,0.218182,0.266667,0.293333,0.333333,0.343750,0.230769,0.266667,0.307692,0.276923,0.293333,0.293333,0.285714,0.321429,0.321429,0.333333,0.321429


showing result for directory de_llama3.1/gemini_2neg_de_fl_senti+_temp_0_no_space
count of bridges ↑


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.150000,0.200000,0.171429,0.254545,0.133333,0.133333,0.150000,0.150000,0.150000,0.171429,0.187500,0.187500,0.200000,0.200000,0.133333,0.133333,0.133333,0.171429,0.171429,0.187500
1,0.200000,0.150000,0.210000,0.200000,0.187500,0.200000,0.085714,0.142857,0.266667,0.266667,0.210000,0.240000,0.187500,0.272727,0.323077,0.307692,0.350000,0.342857,0.300000,0.254545
2,0.150000,0.222222,0.085714,0.087500,0.272727,0.254545,0.171429,0.272727,0.266667,0.133333,0.142857,0.133333,0.066667,0.133333,0.080000,0.000020,0.083333,0.050000,0.000010,0.000010
3,0.000020,0.150000,0.142857,0.150000,0.000020,0.087500,0.150000,0.200000,0.142857,0.142857,0.163636,0.218182,0.218182,0.254545,0.210000,0.254545,0.225000,0.266667,0.240000,0.254545
4,0.200000,0.090909,0.142857,0.166667,0.225000,0.276923,0.342857,0.375000,0.411765,0.225000,0.163636,0.218182,0.266667,0.291667,0.291667,0.373333,0.272727,0.350000,0.254545,0.307692
5,0.222222,0.000020,0.083333,0.187500,0.187500,0.250000,0.240000,0.187500,0.150000,0.171429,0.075000,0.120000,0.133333,0.120000,0.083333,0.171429,0.150000,0.050000,0.000020,0.066667
6,0.222222,0.291667,0.200000,0.210000,0.160000,0.200000,0.291667,0.360000,0.321429,0.254545,0.210000,0.266667,0.210000,0.160000,0.210000,0.307692,0.218182,0.200000,0.291667,0.218182
7,0.160000,0.142857,0.222222,0.323077,0.272727,0.200000,0.200000,0.222222,0.400000,0.393750,0.360000,0.350000,0.291667,0.291667,0.222222,0.266667,0.200000,0.218182,0.150000,0.254545
8,0.085714,0.210000,0.210000,0.083333,0.085714,0.155556,0.155556,0.155556,0.222222,0.250000,0.272727,0.240000,0.272727,0.291667,0.342857,0.291667,0.323077,0.272727,0.291667,0.222222
9,0.087500,0.210000,0.307692,0.155556,0.240000,0.291667,0.300000,0.300000,0.200000,0.243750,0.240000,0.163636,0.218182,0.235714,0.307692,0.300000,0.291667,0.342857,0.254545,0.200000


showing result for directory de_llama3.1/gemini_sent_2neg_de_fl_senti+_temp_0
count of bridges ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,3,4,4,4,3,3,4,5,4,5,6,6,4,4,4,6,6,6,6,5
1,2,4,3,3,4,5,5,5,4,4,3,3,2,2,2,2,2,2,4,4
2,2,2,4,5,3,5,3,4,4,4,4,3,3,4,4,5,5,5,4,4
3,1,1,2,3,3,2,2,3,2,2,2,2,2,2,3,3,3,3,3,3
4,0,3,3,4,3,4,3,3,4,4,4,3,2,3,5,4,4,5,3,3
5,1,6,4,5,5,5,4,3,2,2,3,3,2,3,3,3,2,2,2,3
6,2,3,5,3,3,3,3,3,3,2,2,2,2,2,3,2,1,2,3,4
7,3,3,3,5,4,5,5,4,3,4,4,5,3,4,4,4,6,6,6,5
8,2,4,4,3,4,5,4,4,5,5,5,6,6,6,5,5,6,6,4,5
9,2,2,3,3,2,4,4,3,4,5,4,5,3,4,3,2,3,4,4,5


harmonic mean ↑
fluency ignored


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.171429,0.222222,0.222222,0.240000,0.171429,0.187500,0.222222,0.222222,0.171429,0.222222,0.300000,0.300000,0.266667,0.266667,0.254545,0.323077,0.342857,0.375000,0.375000,0.321429
1,0.142857,0.240000,0.210000,0.187500,0.266667,0.307692,0.307692,0.321429,0.285714,0.276923,0.218182,0.218182,0.155556,0.155556,0.155556,0.160000,0.163636,0.169231,0.276923,0.254545
2,0.150000,0.160000,0.285714,0.291667,0.225000,0.321429,0.218182,0.240000,0.222222,0.200000,0.171429,0.150000,0.171429,0.254545,0.171429,0.187500,0.187500,0.222222,0.200000,0.171429
3,0.083333,0.080000,0.150000,0.210000,0.210000,0.163636,0.160000,0.210000,0.150000,0.150000,0.150000,0.120000,0.142857,0.155556,0.200000,0.210000,0.187500,0.187500,0.187500,0.171429
4,0.000020,0.210000,0.200000,0.254545,0.230769,0.266667,0.218182,0.230769,0.285714,0.266667,0.254545,0.150000,0.142857,0.187500,0.250000,0.222222,0.240000,0.272727,0.210000,0.210000
5,0.085714,0.240000,0.240000,0.291667,0.307692,0.291667,0.266667,0.225000,0.166667,0.169231,0.230769,0.235714,0.166667,0.230769,0.225000,0.225000,0.166667,0.163636,0.160000,0.200000
6,0.142857,0.171429,0.250000,0.225000,0.225000,0.187500,0.200000,0.218182,0.225000,0.160000,0.155556,0.160000,0.166667,0.171429,0.240000,0.169231,0.092308,0.169231,0.218182,0.285714
7,0.187500,0.171429,0.210000,0.272727,0.240000,0.250000,0.222222,0.266667,0.225000,0.240000,0.254545,0.222222,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,0.150000,0.222222
8,0.150000,0.222222,0.222222,0.200000,0.240000,0.272727,0.266667,0.254545,0.307692,0.307692,0.321429,0.342857,0.342857,0.342857,0.307692,0.250000,0.272727,0.240000,0.222222,0.222222
9,0.142857,0.142857,0.171429,0.187500,0.133333,0.200000,0.254545,0.230769,0.285714,0.333333,0.266667,0.333333,0.225000,0.222222,0.225000,0.155556,0.210000,0.240000,0.222222,0.142857


## bridges

In [65]:
bridge_stats(dirs_bridge_llama3_1)

showing result for directory de_llama3.1/gemini_bridge_de_fl_bridge+
count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,13,14,13,16,16,17,19,20,20,20,20,20,20,19,20,20,20,20,19,20
1,14,17,17,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20
2,15,15,14,16,17,20,20,18,20,20,20,20,20,20,20,20,20,20,20,20
3,14,14,9,11,13,12,16,14,16,14,15,17,19,16,18,18,19,19,19,20
4,13,12,13,15,12,13,14,14,15,11,16,18,17,18,17,19,19,19,19,20
5,15,12,13,13,13,13,13,15,13,14,19,18,17,18,19,20,20,20,20,20
6,14,15,14,13,11,10,13,13,17,13,14,16,13,12,15,17,17,17,19,20
7,15,15,13,12,14,15,11,16,13,15,17,16,17,16,15,14,13,13,11,13
8,15,15,10,15,14,13,14,12,14,16,15,14,15,15,15,16,15,16,15,16
9,16,15,15,13,12,14,13,12,11,11,14,12,14,16,16,15,17,16,14,13


average perplexity of continuations ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2.99,3.53,3.27,3.59,3.60,3.43,3.86,3.57,3.80,4.13,3.69,3.45,3.34,3.39,3.33,3.21,3.48,3.69,3.72,4.15
1,3.07,2.95,3.16,3.10,2.49,2.18,2.47,1.86,1.85,1.88,1.82,1.84,1.81,1.82,1.80,1.78,1.75,1.80,2.13,1.93
2,2.67,3.07,3.20,3.10,3.22,2.75,3.15,3.02,2.44,2.38,2.47,2.16,2.49,2.31,3.34,3.02,2.79,2.53,2.76,2.66
3,2.79,3.12,3.54,3.53,3.17,3.21,3.32,3.21,3.08,3.84,3.95,5.40,4.79,4.41,5.94,4.86,6.70,6.98,5.84,5.45
4,3.10,3.09,3.15,3.04,3.14,3.09,2.86,3.11,3.15,3.50,3.61,3.79,4.19,3.84,3.38,3.88,3.50,3.36,3.59,3.81
5,2.94,2.89,2.92,2.95,2.95,2.87,2.85,3.07,3.43,3.58,3.58,3.32,3.70,3.67,3.59,3.36,3.38,3.72,3.62,3.60
6,2.88,2.93,2.88,3.05,3.15,2.96,3.13,3.39,3.43,3.43,3.38,3.79,3.69,3.35,3.39,3.40,3.58,3.53,4.11,4.09
7,3.11,2.93,2.87,3.03,3.10,3.09,3.35,3.16,3.41,3.22,3.18,3.29,3.24,3.43,3.52,3.51,3.56,3.49,3.50,3.59
8,2.90,3.10,3.20,2.97,2.98,3.06,3.30,3.72,3.61,3.58,3.45,3.24,3.08,3.10,3.05,2.97,3.26,3.31,3.44,3.77
9,2.92,3.00,3.00,3.31,3.34,2.99,3.22,3.56,3.36,3.81,3.81,3.95,3.84,3.52,3.49,3.58,3.33,3.41,3.92,3.80


count of sentences talking about the bridge ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2,3,4,6,1,3,2,4,3,4,2,6,4,6,5,8,6,5,4,3
1,1,3,3,4,3,1,4,1,1,1,0,0,0,0,0,0,0,0,2,1
2,0,3,6,6,1,0,0,2,3,5,4,3,6,3,5,2,7,6,4,6
3,0,4,7,5,0,0,0,2,1,1,2,2,1,1,3,11,12,9,12,11
4,0,1,1,0,1,0,2,3,2,3,7,7,7,7,6,8,8,11,12,14
5,0,2,0,1,1,0,0,1,4,7,9,13,15,13,13,17,17,17,16,17
6,0,1,0,1,1,0,0,2,2,3,4,5,7,5,6,9,9,11,9,11
7,0,0,0,0,0,0,0,0,1,0,1,1,1,1,1,3,4,4,3,5
8,0,1,0,0,0,0,0,1,1,2,3,4,4,6,5,5,6,7,7,8
9,0,2,0,0,0,0,0,1,1,1,1,3,4,4,5,6,6,6,9,10


harmonic mean ↑
fluency ignored


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.155556,0.200000,0.254545,0.240000,0.080000,0.150000,0.066667,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.085714,0.000020,0.000020,0.000020,0.000020,0.080000,0.000020
1,0.085714,0.150000,0.150000,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000010,0.000010,0.000010,0.000010,0.000010,0.000010,0.000010,0.000010,0.000020,0.000020
2,0.000020,0.187500,0.300000,0.240000,0.075000,0.000010,0.000010,0.100000,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020
3,0.000020,0.240000,0.427778,0.321429,0.000020,0.000020,0.000020,0.150000,0.080000,0.085714,0.142857,0.120000,0.050000,0.080000,0.120000,0.169231,0.092308,0.090000,0.092308,0.000020
4,0.000020,0.088889,0.087500,0.000020,0.088889,0.000020,0.150000,0.200000,0.142857,0.225000,0.254545,0.155556,0.210000,0.155556,0.200000,0.088889,0.088889,0.091667,0.092308,0.000020
5,0.000020,0.160000,0.000020,0.087500,0.087500,0.000020,0.000020,0.083333,0.254545,0.323077,0.090000,0.173333,0.250000,0.173333,0.092857,0.000020,0.000020,0.000020,0.000020,0.000020
6,0.000020,0.083333,0.000020,0.087500,0.090000,0.000020,0.000020,0.155556,0.120000,0.210000,0.240000,0.222222,0.350000,0.307692,0.272727,0.225000,0.225000,0.235714,0.090000,0.000020
7,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.087500,0.000020,0.075000,0.080000,0.075000,0.080000,0.083333,0.200000,0.254545,0.254545,0.225000,0.291667
8,0.000020,0.083333,0.000020,0.000020,0.000020,0.000020,0.000020,0.088889,0.085714,0.133333,0.187500,0.240000,0.222222,0.272727,0.250000,0.222222,0.272727,0.254545,0.291667,0.266667
9,0.000020,0.142857,0.000020,0.000020,0.000020,0.000020,0.000020,0.088889,0.090000,0.090000,0.085714,0.218182,0.240000,0.200000,0.222222,0.272727,0.200000,0.240000,0.360000,0.411765
